# 01 · RAG Pipeline

**Retrieval-Augmented Generation (RAG)** grounds the LLM's answers in real studio knowledge instead of relying
on its general training data — which knows nothing about BrixoAI specifically.

## Architecture

```
┌─────────────────────────────────────────────────────────────┐
│                  INDEXING STAGE  (once at startup)          │
│                                                             │
│  data/*.md ──► TextLoader ──► RecursiveCharacterTextSplitter│
│                               (500 chars, 60 overlap)       │
│                     │                                       │
│                     ▼                                       │
│           HuggingFace all-MiniLM-L6-v2                      │
│           (384-dim dense embeddings)                        │
│                     │                                       │
│                     ▼                                       │
│             ChromaDB vector store                           │
│             (persisted to chroma_db/)                       │
└─────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────┐
│                  QUERY STAGE  (every chat request)          │
│                                                             │
│  user query ──► embed ──► MMR search ──► top-k chunks       │
│                                    │                        │
│                                    ▼                        │
│                          inject as SystemMessage            │
│                          into LLM prompt                    │
└─────────────────────────────────────────────────────────────┘
```

## Why these choices?

| Decision | Rationale |
|---|---|
| `all-MiniLM-L6-v2` | Compact (384-dim), fast, strong semantic similarity, free via HF Inference API |
| `chunk_size=500` | Small enough for precise retrieval, large enough for full sentences |
| `chunk_overlap=60` | Prevents a concept split across chunk boundaries from being lost |
| ChromaDB | Lightweight, zero-infra, persisted to disk — no external vector DB needed |
| MMR search | Returns diverse chunks — avoids all k results being from the same paragraph |

In [1]:
# Install dependencies (run once)
# %pip install -q langchain langchain-community langchain-chroma langchain-huggingface \
#                langchain-text-splitters chromadb python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(dotenv_path=Path('..') / '.env')

DATA_DIR   = Path('..') / 'data'
CHROMA_DIR = Path('..') / 'chroma_db_nb'  # separate store for this notebook

print('Data files:', [f.name for f in DATA_DIR.glob('*.md')])

Data files: ['faq.md', 'projects.md', 'services.md', 'studio.md']


## Step 1 — Load Documents

In [2]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

loader = DirectoryLoader(
    str(DATA_DIR),
    glob='**/*.md',
    loader_cls=TextLoader,
    loader_kwargs={'encoding': 'utf-8'},
    show_progress=False,
)
documents = loader.load()

print(f'Loaded {len(documents)} documents')
for doc in documents:
    print(f'  {Path(doc.metadata["source"]).name} — {len(doc.page_content)} chars')

Loaded 4 documents
  faq.md — 5487 chars
  projects.md — 3044 chars
  services.md — 2354 chars
  studio.md — 1543 chars


## Step 2 — Split into Chunks

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=60,
    separators=['\n\n', '\n', '. ', ' '],
)
chunks = splitter.split_documents(documents)

print(f'Split into {len(chunks)} chunks')
print('\n--- Sample chunk ---')
print(chunks[0].page_content)

Split into 36 chunks

--- Sample chunk ---
# NexaLab — FAQ

## Pricing & Budget

**How much does a project cost?**
Every project is different, so we scope each one individually before quoting. Pricing depends on the scope, timeline, and complexity. We're transparent throughout — no surprise invoices. To get a quote, book a discovery call or use the "Scope a Project" chat mode.


## Step 3 — Embed & Step 4 — Build ChromaDB

In [4]:
from langchain_huggingface import HuggingFaceEndpointEmbeddings
from langchain_chroma import Chroma

EMBED_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'
hf_token = os.getenv('HF_TOKEN') or None

embeddings = HuggingFaceEndpointEmbeddings(
    model=EMBED_MODEL,
    **(({'huggingfacehub_api_token': hf_token}) if hf_token else {}),
)

print('Building vector store …')
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=str(CHROMA_DIR),
)
print(f'Vector store built — {vectorstore._collection.count()} vectors stored')

c:\Users\User\Documents\WebDev\BrixoAI\capstone-gen-ai\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Building vector store …
Vector store built — 36 vectors stored


## Step 5+6 — Query and Retrieve

In [5]:
retriever = vectorstore.as_retriever(
    search_type='mmr',
    search_kwargs={'k': 4, 'fetch_k': 12},
)

query = 'What AI services does BrixoAI offer?'
results = retriever.invoke(query)

print(f'Query: "{query}"')
print(f'Retrieved {len(results)} chunks:\n')
for i, doc in enumerate(results, 1):
    print(f'[{i}] Source: {Path(doc.metadata["source"]).name}')
    print(doc.page_content[:300])
    print('---')

Query: "What AI services does BrixoAI offer?"
Retrieved 4 chunks:

[1] Source: faq.md
**What AI models power BrixoAI?**
The chat is powered by Llama 3.3 70B via Groq. Studio knowledge is retrieved from a local vector database (ChromaDB) using sentence-transformer embeddings — no external vector DB API calls. Vision analysis uses Llama 4 Scout (Groq multimodal).
---
[2] Source: projects.md
### Key Features
- AI-generated weekly goal plans personalised per user
- Daily check-ins with sentiment analysis and streak tracking
- Smart retrospectives — AI-written summaries at the end of each week
- Team workspace with shared goal boards
- Multi-language support (EN, ES, FR, DE)
- Full Stripe
---
[3] Source: projects.md
### Overview
Orbitly is a full-featured, AI-powered productivity platform that helps individuals and small teams set, track, and achieve goals. Users get personalised weekly plans, smart progress nudges, retrospective summaries, and an AI coach that adapts recommendations based 

## Evaluation: Retrieval Quality

We test three queries and inspect whether the retrieved chunks are topically relevant.

In [ ]:
test_queries = [
    'What is the Orbitly project?',
    'How does BrixoAI handle data analytics?',
    'How much does a website cost?',
]

for q in test_queries:
    docs = retriever.invoke(q)
    sources = [Path(d.metadata['source']).name for d in docs]
    print(f'Q: {q}')
    print(f'  Sources: {sources}')
    print(f'  Top snippet: {docs[0].page_content[:200]}\n')

Q: What is the Orbitly project?
  Sources: ['faq.md', 'faq.md', 'services.md', 'projects.md']
  Top snippet: **What AI models do you use?**
Primarily Groq (Llama 3, Deepseek), OpenAI (GPT-4o, embeddings), and Anthropic (Claude). We choose based on cost, speed, and capability requirements per project.

---

#

Q: How does BrixoAI handle data analytics?
  Sources: ['faq.md', 'faq.md', 'services.md', 'projects.md']
  Top snippet: **How does the data analysis feature work?**
Upload a CSV or Excel file (up to 5 MB) in Data Insights mode. The backend parses it with pandas, builds column profiles, computes statistics (distribution

Q: How much does a website cost?
  Sources: ['faq.md', 'services.md', 'faq.md', 'projects.md']
  Top snippet: **Do you work with fixed budgets?**
Yes. If you share your budget upfront, we'll scope the project to fit it and be honest if it's not feasible rather than overpromising.

**What are typical price ran



## Summary

- **Indexing** is O(n·d) where n = chunks and d = embedding dimension.
- **Query time** is sub-millisecond for cosine similarity with ChromaDB.
- **MMR** improves diversity — especially important when studio.md and faq.md overlap on the same topics.
- The retriever is stateless — safe to share across concurrent requests.